# Saber Post-BERM Sweep (31 tasks) — GSM8K Full

目标：在 `saber_expand=True, berm_mode=cross_step, saber_global_aadu=True, saber_mtr=0.8` 基础上，探索两条方向：
1) 结束后额外 1-2 轮 global BERM + temp block repair；
2) 在 warm-up/rewarm（full-forward 阶段）也启用 BERM。

并加入一致性 baseline：`n=4, mu=8, bl=32`（不开方向1/2）。

## 1. 环境设置

In [ ]:
import os, gc, re, json, datetime, threading, queue, subprocess
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt

os.environ['CUDA_VISIBLE_DEVICES'] = os.environ.get('CUDA_VISIBLE_DEVICES', '4,5')
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

os.chdir('llada')
os.makedirs('nlogs', exist_ok=True)
os.makedirs('evals_results/saber', exist_ok=True)

torch.cuda.empty_cache()
gc.collect()
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())

## 2. 任务配置

In [ ]:
task = 'gsm8k'
fewshot = 5
seed = 42
gen_length = 256
steps = 256
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')


def _saber_task(name, n, mu, bl, post_rounds=0, post_wmul=1, fullstage=False):
    extra_args = [
        'saber_expand=True',
        'berm_mode=cross_step',
        'saber_global_aadu=True',
        'saber_mtr=0.8',
        f'saber_n={n}',
        f'saber_mu={mu}',
        f'block_length={bl}',
        f'saber_post_global_berm_rounds={post_rounds}',
        f'saber_post_global_berm_window_mul={post_wmul}',
        f'saber_fullstage_berm={str(fullstage)}',
    ]
    return {'name': name, 'extra_args': extra_args}


TASK_CONFIGS = [
    _saber_task('saber_exp_gl_cs_n4_mu8_bl32_base', 4, 8, 32, post_rounds=0, post_wmul=1, fullstage=False),

    _saber_task('saber_p1_n4_mu8_r1_w1_bl32', 4, 8, 32, 1, 1, False),
    _saber_task('saber_p1_n4_mu8_r1_w2_bl32', 4, 8, 32, 1, 2, False),
    _saber_task('saber_p1_n4_mu8_r2_w1_bl32', 4, 8, 32, 2, 1, False),
    _saber_task('saber_p1_n4_mu8_r2_w2_bl32', 4, 8, 32, 2, 2, False),
    _saber_task('saber_p1_n5_mu8_r1_w1_bl32', 5, 8, 32, 1, 1, False),
    _saber_task('saber_p1_n5_mu8_r2_w1_bl32', 5, 8, 32, 2, 1, False),
    _saber_task('saber_p1_n4_mu6_r1_w1_bl32', 4, 6, 32, 1, 1, False),
    _saber_task('saber_p1_n4_mu12_r1_w1_bl32', 4, 12, 32, 1, 1, False),
    _saber_task('saber_p1_n6_mu8_r1_w1_bl32', 6, 8, 32, 1, 1, False),
    _saber_task('saber_p1_n6_mu8_r2_w1_bl32', 6, 8, 32, 2, 1, False),

    _saber_task('saber_p2_n6_mu6_bl32', 6, 6, 32, 0, 1, True),
    _saber_task('saber_p2_n6_mu8_bl32', 6, 8, 32, 0, 1, True),
    _saber_task('saber_p2_n6_mu12_bl32', 6, 12, 32, 0, 1, True),
    _saber_task('saber_p2_n8_mu6_bl32', 8, 6, 32, 0, 1, True),
    _saber_task('saber_p2_n8_mu8_bl32', 8, 8, 32, 0, 1, True),
    _saber_task('saber_p2_n8_mu12_bl32', 8, 12, 32, 0, 1, True),
    _saber_task('saber_p2_n10_mu6_bl32', 10, 6, 32, 0, 1, True),
    _saber_task('saber_p2_n10_mu8_bl32', 10, 8, 32, 0, 1, True),
    _saber_task('saber_p2_n10_mu12_bl32', 10, 12, 32, 0, 1, True),
    _saber_task('saber_p2_n12_mu8_bl32', 12, 8, 32, 0, 1, True),
    _saber_task('saber_p2_n12_mu12_bl32', 12, 12, 32, 0, 1, True),
    _saber_task('saber_p2_n14_mu8_bl32', 14, 8, 32, 0, 1, True),
    _saber_task('saber_p2_n8_mu8_bl64', 8, 8, 64, 0, 1, True),

    _saber_task('saber_p12_n6_mu8_r1_w1_bl32', 6, 8, 32, 1, 1, True),
    _saber_task('saber_p12_n8_mu8_r1_w1_bl32', 8, 8, 32, 1, 1, True),
    _saber_task('saber_p12_n10_mu8_r1_w1_bl32', 10, 8, 32, 1, 1, True),
    _saber_task('saber_p12_n8_mu12_r1_w2_bl32', 8, 12, 32, 1, 2, True),
    _saber_task('saber_p12_n10_mu12_r1_w2_bl32', 10, 12, 32, 1, 2, True),
    _saber_task('saber_p12_n12_mu8_r2_w1_bl32', 12, 8, 32, 2, 1, True),
    _saber_task('saber_p12_n8_mu8_r2_w1_bl64', 8, 8, 64, 2, 1, True),
]

assert len(TASK_CONFIGS) == 31
GPU_POOL = [int(x) for x in os.environ['CUDA_VISIBLE_DEVICES'].split(',')]
print('Total tasks:', len(TASK_CONFIGS), '| GPUs:', GPU_POOL, '| timestamp:', timestamp)

## 3. 并行启动任务

In [ ]:
task_queue = queue.Queue()
for cfg in TASK_CONFIGS:
    task_queue.put(cfg)

results_lock = threading.Lock()
all_results = []


def gpu_worker(gpu_id):
    while True:
        try:
            cfg = task_queue.get_nowait()
        except queue.Empty:
            return

        name = cfg['name']
        log_file = f'nlogs/sweep_saber_postberm_{task}_{name}_{timestamp}.log'
        output_dir = f'evals_results/saber/{task}-{name}-{timestamp}'

        common_args = [
            "model_path='GSAI-ML/LLaDA-8B-Instruct'",
            f'gen_length={gen_length}',
            f'steps={steps}',
            'show_speed=True',
            f'seed={seed}',
        ]
        model_args = ','.join(common_args + cfg['extra_args'])
        cmd = (
            f'CUDA_VISIBLE_DEVICES={gpu_id} accelerate launch eval_llada.py '
            f'--tasks {task} --num_fewshot {fewshot} --confirm_run_unsafe_code '
            f'--model llada_dist --model_args {model_args} --output_path {output_dir} --log_samples'
        )

        print(f'[GPU {gpu_id}] START {name}')
        p = subprocess.Popen(
            cmd,
            shell=True,
            stdout=open(log_file, 'w'),
            stderr=subprocess.STDOUT,
        )
        rc = p.wait()
        print(f'[GPU {gpu_id}] DONE  {name} rc={rc}')
        with results_lock:
            all_results.append((cfg, name, log_file, output_dir, rc))
        task_queue.task_done()


threads = []
for gid in GPU_POOL:
    t = threading.Thread(target=gpu_worker, args=(gid,), daemon=True)
    t.start()
    threads.append(t)

for t in threads:
    t.join()

print('All finished:', len(all_results), '/', len(TASK_CONFIGS))

## 4. 解析评测结果

In [ ]:
def parse_result(cfg, output_dir, log_file):
    name = cfg['name']
    bl_m = re.search(r'bl(\d+)', name)
    n_m = re.search(r'_n(\d+)', name)
    mu_m = re.search(r'_mu(\d+)', name)
    r_m = re.search(r'_r(\d+)', name)
    w_m = re.search(r'_w(\d+)', name)

    result_json = Path(output_dir) / 'results.json'
    flex_acc = None
    strict_acc = None
    if result_json.exists():
        data = json.loads(result_json.read_text(encoding='utf-8'))
        metrics = data.get('results', {}).get(task, {})
        flex_acc = metrics.get('exact_match,flexible-extract')
        strict_acc = metrics.get('exact_match,strict-match')

    log_content = Path(log_file).read_text(encoding='utf-8', errors='ignore') if Path(log_file).exists() else ''

    if flex_acc is None:
        flex_m2 = re.search(r'flexible-extract.*?exact_match.*?([\d.]+)', log_content)
        if flex_m2:
            flex_acc = float(flex_m2.group(1))
    if strict_acc is None:
        strict_m2 = re.search(r'strict-match.*?exact_match.*?([\d.]+)', log_content)
        if strict_m2:
            strict_acc = float(strict_m2.group(1))

    speed_m = re.search(r'Tokens per second:\s*([\d.]+)', log_content)
    if speed_m is None:
        speed_m = re.search(r'Average generation speed:\s*([\d.]+)', log_content)
    if speed_m is None:
        speed_m = re.search(r'tok/s[:\s]+([\d.]+)', log_content)
    nfe_m = re.search(r'Total NFE is (\d+)', log_content)
    tok_m = re.search(r'Total number of tokens generated:\s*(\d+)', log_content)
    if tok_m is None:
        tok_m = re.search(r'Total tokens generated:\s*(\d+)', log_content)
    time_m = re.search(r'Total time taken:\s*([\d.]+)', log_content)
    if time_m is None:
        time_m = re.search(r'Total generation time:\s*([\d.]+)', log_content)

    return {
        'name': name,
        'n': int(n_m.group(1)) if n_m else None,
        'mu': int(mu_m.group(1)) if mu_m else None,
        'bl': int(bl_m.group(1)) if bl_m else None,
        'post_rounds': int(r_m.group(1)) if r_m else 0,
        'post_wmul': int(w_m.group(1)) if w_m else 1,
        'fullstage': int('_p2_' in name or '_p12_' in name),
        'flex_acc': float(flex_acc) if flex_acc is not None else None,
        'strict_acc': float(strict_acc) if strict_acc is not None else None,
        'tok_per_sec': float(speed_m.group(1)) if speed_m else None,
        'total_nfe': int(nfe_m.group(1)) if nfe_m else None,
        'total_tokens': int(tok_m.group(1)) if tok_m else None,
        'time_sec': float(time_m.group(1)) if time_m else None,
    }


rows = []
for cfg, name, log_file, output_dir, rc in all_results:
    rows.append(parse_result(cfg, output_dir, log_file))

df = pd.DataFrame(rows).sort_values(['bl', 'post_rounds', 'fullstage', 'n', 'mu', 'post_wmul', 'name'])
pd.set_option('display.max_rows', 200)
display(df)

print('\nTop by FlexAcc:')
display(df.sort_values('flex_acc', ascending=False).head(10))

## 5. 对比图：Accuracy / NFE / Speed

In [ ]:
plot_df = df.copy()
plot_df = plot_df[plot_df['flex_acc'].notna()]
plot_df['label'] = plot_df['name']

fig, axes = plt.subplots(3, 1, figsize=(18, 13), sharex=True)
metrics = [('flex_acc', 'FlexAcc'), ('total_nfe', 'Total NFE'), ('tok_per_sec', 'Tokens/sec')]
anchor_name = 'saber_exp_gl_cs_n4_mu8_bl32_base'
colors = ['#E53935' if n == anchor_name else '#4E79A7' for n in plot_df['name']]

for ax, (col, title) in zip(axes, metrics):
    vals = plot_df[col].fillna(0)
    ax.bar(plot_df['label'], vals, color=colors)
    ax.set_title(title)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

axes[-1].tick_params(axis='x', rotation=75, labelsize=8)
plt.tight_layout()
plt.show()

## 6. 从已有结果重新加载（可选）

如果 notebook 内核重启，运行此 cell 从磁盘加载已有结果（会自动检测最新 timestamp），然后重新运行 Section 5 生成图表。

In [ ]:
import glob, re, json, os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

task = 'gsm8k'
seed = 42

TASK_CONFIGS = [
    {'name': 'saber_exp_gl_cs_n4_mu8_bl32_base'},
    {'name': 'saber_p1_n4_mu8_r1_w1_bl32'},
    {'name': 'saber_p1_n4_mu8_r1_w2_bl32'},
    {'name': 'saber_p1_n4_mu8_r2_w1_bl32'},
    {'name': 'saber_p1_n4_mu8_r2_w2_bl32'},
    {'name': 'saber_p1_n5_mu8_r1_w1_bl32'},
    {'name': 'saber_p1_n5_mu8_r2_w1_bl32'},
    {'name': 'saber_p1_n4_mu6_r1_w1_bl32'},
    {'name': 'saber_p1_n4_mu12_r1_w1_bl32'},
    {'name': 'saber_p1_n6_mu8_r1_w1_bl32'},
    {'name': 'saber_p1_n6_mu8_r2_w1_bl32'},
    {'name': 'saber_p2_n6_mu6_bl32'},
    {'name': 'saber_p2_n6_mu8_bl32'},
    {'name': 'saber_p2_n6_mu12_bl32'},
    {'name': 'saber_p2_n8_mu6_bl32'},
    {'name': 'saber_p2_n8_mu8_bl32'},
    {'name': 'saber_p2_n8_mu12_bl32'},
    {'name': 'saber_p2_n10_mu6_bl32'},
    {'name': 'saber_p2_n10_mu8_bl32'},
    {'name': 'saber_p2_n10_mu12_bl32'},
    {'name': 'saber_p2_n12_mu8_bl32'},
    {'name': 'saber_p2_n12_mu12_bl32'},
    {'name': 'saber_p2_n14_mu8_bl32'},
    {'name': 'saber_p2_n8_mu8_bl64'},
    {'name': 'saber_p12_n6_mu8_r1_w1_bl32'},
    {'name': 'saber_p12_n8_mu8_r1_w1_bl32'},
    {'name': 'saber_p12_n10_mu8_r1_w1_bl32'},
    {'name': 'saber_p12_n8_mu12_r1_w2_bl32'},
    {'name': 'saber_p12_n10_mu12_r1_w2_bl32'},
    {'name': 'saber_p12_n12_mu8_r2_w1_bl32'},
    {'name': 'saber_p12_n8_mu8_r2_w1_bl64'},
]

first_name = TASK_CONFIGS[0]['name']
latest_logs = sorted(
    glob.glob(f'nlogs/sweep_saber_postberm_{task}_{first_name}_*.log'),
    key=os.path.getmtime, reverse=True,
)
if latest_logs:
    fname = os.path.basename(latest_logs[0])
    timestamp = fname.replace(f'sweep_saber_postberm_{task}_{first_name}_', '').replace('.log', '')
    print(f'Auto-detected latest timestamp: {timestamp}')
else:
    timestamp = 'NOTFOUND'
    print('WARNING: No log found!')

parsed_results = []
for cfg in TASK_CONFIGS:
    name = cfg['name']
    log_file = f'nlogs/sweep_saber_postberm_{task}_{name}_{timestamp}.log'
    output_dir = f'evals_results/saber/{task}-{name}-{timestamp}'

    if not os.path.exists(log_file):
        print(f'  [MISS] {name}')
        continue

    with open(log_file, 'r') as f:
        content = f.read()

    result_json = Path(output_dir) / 'results.json'
    flex_acc = None
    strict_acc = None
    if result_json.exists():
        data = json.loads(result_json.read_text(encoding='utf-8'))
        metrics = data.get('results', {}).get(task, {})
        flex_acc = metrics.get('exact_match,flexible-extract')
        strict_acc = metrics.get('exact_match,strict-match')

    if flex_acc is None:
        flex_m = re.search(r'flexible-extract.*?exact_match.*?([\d.]+)', content)
        if flex_m:
            flex_acc = float(flex_m.group(1))
    if strict_acc is None:
        strict_m = re.search(r'strict-match.*?exact_match.*?([\d.]+)', content)
        if strict_m:
            strict_acc = float(strict_m.group(1))

    speed_m = re.search(r'Tokens per second:\s*([\d.]+)', content)
    if speed_m is None:
        speed_m = re.search(r'Average generation speed:\s*([\d.]+)', content)
    if speed_m is None:
        speed_m = re.search(r'tok/s[:\s]+([\d.]+)', content)
    nfe_m = re.search(r'Total NFE is (\d+)', content)
    tok_m = re.search(r'Total number of tokens generated:\s*(\d+)', content)
    if tok_m is None:
        tok_m = re.search(r'Total tokens generated:\s*(\d+)', content)
    time_m = re.search(r'Total time taken:\s*([\d.]+)', content)
    if time_m is None:
        time_m = re.search(r'Total generation time:\s*([\d.]+)', content)

    bl_m = re.search(r'bl(\d+)', name)
    n_m = re.search(r'_n(\d+)', name)
    mu_m = re.search(r'_mu(\d+)', name)
    r_m = re.search(r'_r(\d+)', name)
    w_m = re.search(r'_w(\d+)', name)

    parsed_results.append({
        'name': name,
        'n': int(n_m.group(1)) if n_m else None,
        'mu': int(mu_m.group(1)) if mu_m else None,
        'bl': int(bl_m.group(1)) if bl_m else None,
        'post_rounds': int(r_m.group(1)) if r_m else 0,
        'post_wmul': int(w_m.group(1)) if w_m else 1,
        'fullstage': int('_p2_' in name or '_p12_' in name),
        'flex_acc': float(flex_acc) if flex_acc is not None else None,
        'strict_acc': float(strict_acc) if strict_acc is not None else None,
        'tok_per_sec': float(speed_m.group(1)) if speed_m else None,
        'total_nfe': int(nfe_m.group(1)) if nfe_m else None,
        'total_tokens': int(tok_m.group(1)) if tok_m else None,
        'time_sec': float(time_m.group(1)) if time_m else None,
    })
    print(f'  [OK]   {name}')

df = pd.DataFrame(parsed_results).sort_values(['bl', 'post_rounds', 'fullstage', 'n', 'mu', 'post_wmul', 'name'])

print(f'\nLoaded {len(parsed_results)} results  (seed={seed}, timestamp={timestamp})')
print(f'\n{"Name":<42} {"n":<4} {"mu":<5} {"bl":<5} {"FlexAcc":<10} {"StrictAcc":<11} {"Tok/s":<10} {"NFE":<10} {"Time(s)":<8}')
print('-' * 105)
for _, r in df.iterrows():
    n_s = str(int(r['n'])) if r['n'] is not None else '—'
    mu_s = str(int(r['mu'])) if r['mu'] is not None else '—'
    bl_s = str(int(r['bl'])) if r['bl'] is not None else '—'
    fa = f"{r['flex_acc']:.4f}" if r['flex_acc'] is not None else 'N/A'
    sa = f"{r['strict_acc']:.4f}" if r['strict_acc'] is not None else 'N/A'
    sp = f"{r['tok_per_sec']:.1f}" if r['tok_per_sec'] is not None else 'N/A'
    nf = str(int(r['total_nfe'])) if r['total_nfe'] is not None else 'N/A'
    tm = f"{r['time_sec']:.1f}" if r['time_sec'] is not None else 'N/A'
    print(f"{r['name']:<42} {n_s:<4} {mu_s:<5} {bl_s:<5} {fa:<10} {sa:<11} {sp:<10} {nf:<10} {tm:<8}")

print(f'\nRe-run Section 5 to regenerate plots.')